# Reading a satellite image properly

Four questions, in the order that actually settles the problem.  Q1 opens the file and
finds two pictures that are wrong; Q2 explains the first one, Q4 explains the second.

Everything is read with `rasterio`, which is GDAL with a friendlier interface — Colab
already has it.  The file is a THEOS-2 scene over Bangkok.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/readcheck.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'readcheck'):
    sys.modules.pop(m, None)
import numpy as np
import rasterio
from readcheck import *

PATH = urllib.request.urlretrieve(f'{BASE}/dataset/intake/input.tif', 'input.tif')[0]

## Q1.  Am I reading it correctly?

Open it, read it, look at what came back.

In [ ]:
with rasterio.open(PATH) as d:
    img = d.read()                     # (band, row, col)

print(img.shape, img.dtype)
print('values', img.min(), '-', img.max())

An array arrived, with four bands and values from 1 to 4095.  No error.  Now draw it
three ways.

In [ ]:
check_read(PATH)

Only the middle one is right, and the two failures have different causes.

| panel | what is wrong | settled in |
|---|---|---|
| 1 | drawn on a 0 - 65,535 scale, so it is black | **Q2** |
| 3 | bands taken in the opposite order, so water is blue | **Q4** |

Panel 3 is the dangerous one: it raises no error and looks like a real image.

## Q2.  What type are the numbers?

Panel 1 was black because `uint16` was taken at face value.

In [ ]:
with rasterio.open(PATH) as d:
    print('dtype', d.dtypes[0])

print('the box holds 0 -', np.iinfo(img.dtype).max)
print('the data uses 0 -', img.max())

In [ ]:
check_type(PATH)

`uint16` is the size of the box, not the amount in it.  The sensor is 12-bit, so the
values stop at 4095 and 94 % of the box is never used.  Stretch to the range the data
actually occupies and panel 1 becomes panel 2.

That also means **the display range has to be chosen**, every time.  It is not a
property of the file.

## Q3.  How big is it?

Two different numbers answer this.

In [ ]:
with rasterio.open(PATH) as d:
    print('pixels    ', d.width, 'x', d.height, ' bands', d.count)
    print('pixel size', d.res)
    print('bounds    ', tuple(round(v, 1) for v in d.bounds), d.crs)

In [ ]:
check_size(PATH)

How much ground it covers decides whether it is the right scene.  How much memory it
takes decides whether it opens at all — on disk it is 21 MB, in memory 46 MB, because
the file is compressed and the array is not.

## Q4.  How are the bands arranged?

Panel 3 was blue because the band order was guessed.  So ask the file.

In [ ]:
with rasterio.open(PATH) as d:
    print('bands      ', d.count, d.indexes)
    print('names      ', d.descriptions)
    print('colorinterp', tuple(c.name for c in d.colorinterp))

No names, and `colorinterp` says `gray` followed by `undefined` three times.  **The
file does not know which band is which.**  That has to come from the product spec — or
from looking.

In [ ]:
check_bands(PATH)

In the fourth band the river goes black and the parks go bright, which is what water
and vegetation do to near infrared.  So band 4 is NIR and bands 1 - 3 are red, green,
blue — which is what panel 2 of Q1 assumed, and panel 3 did not.